# Value Iteration vs Policy Iteration

Chapter 15: Bellman Equations and Operators · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare convergence speed and policy stability for value iteration and policy iteration.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-15/vi-vs-pi/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's 3-state, 2-action MDP and defaults
gamma, K = 0.92, 18
P = torch.tensor([[[0.9, 0.1, 0.0], [0.2, 0.8, 0.0], [0.1, 0.0, 0.9]],      # action a0
                  [[0.5, 0.5, 0.0], [0.0, 0.4, 0.6], [0.0, 0.2, 0.8]]],     # action a1
                 dtype=torch.float64)
R = torch.tensor([[0.4, 0.5, 0.3],
                  [0.1, 0.9, 1.2]], dtype=torch.float64)                    # R[a, s]
I = torch.eye(3, dtype=torch.float64)


def q_values(V):
    return R + gamma * P @ V                      # shape (actions, states)


def evaluate(pi):
    """Exact V^pi by solving (I - gamma P_pi) V = r_pi."""
    P_pi = P[pi, torch.arange(3)]
    r_pi = R[pi, torch.arange(3)]
    return torch.linalg.solve(I - gamma * P_pi, r_pi)


# ── Value iteration: one max-backup per step ────────────────────────────────
V = torch.zeros(3, dtype=torch.float64)
for k in range(K):
    V = q_values(V).max(0).values
pi_vi = q_values(V).argmax(0)

# ── Policy iteration: evaluate exactly, then act greedily ───────────────────
pi = torch.zeros(3, dtype=torch.long)
for it in range(1, K + 1):
    V_pi = evaluate(pi)
    new_pi = q_values(V_pi).argmax(0)
    print(f"policy iteration step {it}: policy {pi.tolist()} -> {new_pi.tolist()}, V = {[round(v, 4) for v in V_pi.tolist()]}")
    if torch.equal(new_pi, pi):
        break
    pi = new_pi

V_star = evaluate(pi)
print("\noptimal policy (PI):", pi.tolist(), " V* =", [round(v, 4) for v in V_star.tolist()])
print(f"value iteration after {K} steps: policy {pi_vi.tolist()}, ||V_VI - V*||_inf = {(V - V_star).abs().max().item():.4f}")
# VI's error shrinks like gamma^k; PI stops after a few exact solves
print(f"gamma^{K} = {gamma ** K:.4f}; VI needs about {torch.log(torch.tensor(1e-6)).item() / torch.log(torch.tensor(gamma)).item():.0f} steps for 1e-6")

policy iteration step 1: policy [0, 0, 0] -> [0, 0, 1], V = [5.323, 5.6039, 4.5914]
policy iteration step 2: policy [0, 0, 1] -> [0, 1, 1], V = [5.323, 5.6039, 8.4512]
policy iteration step 3: policy [0, 1, 1] -> [1, 1, 1], V = [9.6999, 13.7868, 14.1544]
policy iteration step 4: policy [1, 1, 1] -> [1, 1, 1], V = [11.9295, 13.7868, 14.1544]

optimal policy (PI): [1, 1, 1]  V* = [11.9295, 13.7868, 14.1544]
value iteration after 18 steps: policy [1, 1, 1], ||V_VI - V*||_inf = 3.1350
gamma^18 = 0.2229; VI needs about 166 steps for 1e-6
